# Recruitment assay

**What it does.** Quantify how strongly a marker is recruited to the pathogen relative to the surrounding cytoplasm.

**When to use it.** For host-pathogen imaging where the readout is localisation rather than abundance.

**What you get.** Per-object recruitment ratios and per-condition summary plots.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.submodules.analyze_recruitment`

```
analyze_recruitment(settings)
```

Quantify recruitment of a fluorescent marker to the pathogenic vacuole and produce per-PV / per-well summaries.

In [ ]:
from spacr.submodules import analyze_recruitment

## 3. Settings and API reference

Read the descriptions here, then edit only the values in the next cell. Defaults and descriptions are generated from the installed spaCR version, so the notebook stays aligned with the API.

### [`spacr.submodules.analyze_recruitment`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.analyze_recruitment)

- **`cell_chann_dim`** — (int) - Recruitment analysis only (analyze_recruitment): the image-channel index paired with the cell mask when drawing outline overlays, and the switch that enables the cell filters - set an integer and cell_size_range, cell_intensity_range and target_intensity_min are applied; leave it None and cells are not filtered at all. Default 3.
- **`cell_intensity_range`** — (list) - [min, max] bounds on a cell's mean intensity, applied when the measurement table is filtered during recruitment analysis, and only when cell_chann_dim is set. BEWARE which channel it tests: _object_filter is called with mask_chans ordered [nucleus, pathogen, cell] and index 0, so the column it actually reads is the NUCLEUS channel, not the cell one. Check the filtered counts before trusting it. Default None.
- **`cell_mask_dim`** — (int) - Position along the last axis of each merged/*.npy array where the cell label mask sits. Merged arrays are ordered [image channels..., cell, nucleus, pathogen, organelle], so the default 4 assumes the four channels 0-3 were kept; keep fewer channels and every mask dim shifts down. None makes measure_crop skip all cell measurements and cell crops. Default 4.
- **`cell_plate_metadata`** — (list of lists) - Wells occupied by each entry of cell_types, one inner list per cell type in the same order, e.g. [['c2','c3'],['c4']]. Every identifier must start with 'c' (column) or 'r' (row); anything else is SILENTLY skipped and those wells get no host_cells label. An unlabelled well is not lost -- 'condition' joins whichever labels do exist -- so a typo here quietly changes what is being compared rather than raising. Default None.
- **`cell_size_range`** — (list) - [min, max] bounds in pixels^2 on cell_area, used to drop rows from the measurement table during recruitment analysis; only cells strictly between the two values are kept. Both entries must be integers or that bound is silently skipped. Setting it to None widens it to [0, 1e100]. Default [0, 100000].
- **`cell_types`** — (list) - Names of the host cell lines in the experiment, e.g. ['HeLa']. Each name is written into the host_cells column and folded into the combined condition label used for grouping and plotting; the list is positionally paired with cell_plate_metadata, which says which wells hold each one. Default ['HeLa'].
- **`cells_per_well`** — (int) - Minimum cells a well must contribute to survive recruitment analysis; wells below it, and every cell in them, are dropped before the by-well plots and CSVs are produced. Raise it to suppress noisy, sparsely populated wells at the cost of losing those wells. Default 0, which keeps every well.
- **`channel_dims`** — (list) - Recruitment analysis only: the image-channel indices held in the merged arrays. They drive the overlay figures and the recruitment loop -- but _calculate_recruitment writes fixed, channel-less column names, so changing this list changes WHICH channels are measured without changing what the output columns are called. Two runs with different values produce identically-named columns holding different measurements. Default [0, 1, 2, 3].
- **`channel_of_interest`** — (int) - Index of the fluorescence channel the downstream analysis focuses on. It decides which channel's features survive filtering (other channels' features are dropped), defines recruitment = pathogen_channel_N_mean_intensity / cytoplasm_channel_N_mean_intensity, and is written into the ML result paths. Set it to the channel carrying your phenotype readout. Valid 0-3; default 3 in the ML/recruitment steps, 1-2 elsewhere.
- **`figuresize`** — (int) - Base figure size in inches; figures are built square as figuresize x figuresize and font sizes are derived from it (legend, axis labels and ticks at 0.75x, overlay text at 0.5x). Raise it when text is unreadable at publication scale, lower it to fit panels on screen. Default 10; cluster grids cap total width at 200 inches.
- **`nuclei_limit`** — (int, bool, or None) - Cap on nuclei per cell, applied when the per-object tables are merged. None disables the filter, True keeps only single-nucleus cells, and an integer N keeps cells with N or fewer. Cells over the cap are dropped from the merged table entirely. Do NOT pass False: it is read as 0 and removes every cell, leaving an empty analysis rather than an error. Default None.
- **`nucleus_chann_dim`** — (int) - Recruitment analysis only (analyze_recruitment): the image-channel index paired with the nucleus mask when drawing outline overlays, and the switch that enables nucleus_size_range / nucleus_intensity_range filtering. Set it to None to skip nucleus filtering. It plays no part in segmentation - use nucleus_channel for that. Default 0.
- **`nucleus_intensity_range`** — (list) - Two-element [min, max] bound on mean nucleus-channel intensity used by the recruitment analysis to drop rows from the measurement table - it filters measured objects, not masks or normalization. Rows are kept only if min &lt; mean intensity &lt; max (raw units), and each bound is ignored unless it is an int. Default [0, 100000].
- **`nucleus_mask_dim`** — (int) - Position along the last axis of each merged/*.npy array where the nucleus label mask sits, one plane after the cell mask. With the default four image channels (0-3) that is 5; keep a different number of channels and it shifts by the same amount. None makes measure_crop skip nucleus measurements and cell-to-nucleus linking. Default 5.
- **`nucleus_size_range`** — (list) - Two-element [min, max] bound in pixels^2 on nucleus_area, used by the recruitment analysis to drop rows from the measurement table; masks are left untouched. Rows are kept only if min &lt; area &lt; max, and each bound is ignored unless it is an int. Default [0, 100000]; None widens it to [0, 1e100].
- **`pathogen_chann_dim`** — (int) - Recruitment analysis only (analyze_recruitment): the image-channel index paired with the pathogen mask when drawing outline overlays, and the switch that enables pathogen_size_range / pathogen_intensity_range filtering. Set it to None to skip pathogen filtering. It plays no part in segmentation - use pathogen_channel for that. Default 2.
- **`pathogen_intensity_range`** — (list) - Two-element [min, max] mean-intensity filter applied to the pathogen table in analyze_recruitment; pathogens whose mean intensity in the paired mask channel falls outside the open interval are dropped before recruitment ratios are computed. Bounds must be ints - floats are silently ignored. Default [0, 100000]. Use it to exclude dead or saturated parasites.
- **`pathogen_limit`** — (int, bool, or None) - Maximum pathogens per cell. True or 1 = single pathogen only; None or False = no limit; int = custom limit. Default varies by module (1, 3, 10 or 1000 depending on the factory that fills it), so check the module's own settings rather than assuming one value.
- **`pathogen_mask_dim`** — (int) - Position along the last axis of each merged/*.npy array where the pathogen label mask sits, one plane after the nucleus mask. With the default four image channels (0-3) that is 6; shift it if you keep a different number of channels. None makes measure_crop skip pathogen measurements, so infection status cannot be scored. Default 6.
- **`pathogen_plate_metadata`** — (list of lists) - Well locations of each pathogen condition, one inner list per entry in pathogen_types. Every item must be a row or column ID string such as 'c1' or 'r3'; anything else is silently ignored and those wells stay unannotated. Ranges like 'c2-c11' are not expanded - list each row/column. Do not leave it None while pathogen_types is set: annotation is not skipped, every row is labelled with the first pathogen_types entry. Defaults: None in the plot-from-db settings, [['c1','c2','c3'],['c4','c5','c6']] for recruitment analysis.
- **`pathogen_size_range`** — (list) - Two-element [min, max] area filter in pixels squared applied to the pathogen table in analyze_recruitment, well after segmentation: rows with pathogen_area outside the open interval are dropped. Bounds must be ints - floats are silently ignored. None widens it to effectively unlimited. Default [0, 100000]. Use it to discard debris and merged clumps.
- **`pathogen_types`** — (list) - Names given to each pathogen condition on the plate, e.g. ['wt','ku80']. Element i is written into the pathogen column for every well listed in pathogen_plate_metadata[i] and folded into the combined condition label used for grouping and plotting. Must match pathogen_plate_metadata in length and order; None skips pathogen annotation. Default ['pathogen_1', 'pathogen_2'] for the dataset builders, ['pc'] for the control-based paths, None where types are not used.
- **`plot`** — (bool) - Render and save QC figures while the pipeline runs: channel montages and Cellpose mask overlays during segmentation, before/after filtration views and crop grids during measurement. It adds figures per batch, so a full plate becomes much slower and more memory-hungry; keep it for small or test_mode runs, which force it on. Default False.
- **`plot_control`** — (bool) - Before the recruitment plots, draw a control panel of per-compartment mean intensities (cell, nucleus, pathogen, cytoplasm) for every channel, split by condition. Use it to confirm channel assignment and that positive/negative control wells separate as expected before trusting the recruitment numbers. Turn it off to shorten the run. Default True.
- **`plot_nr`** — (int) - How many merged image stacks from the start of the folder are drawn with cell, nucleus and pathogen outlines overlaid before recruitment analysis runs. The check is index &lt;= plot_nr, so plot_nr + 1 images actually appear and 0 still plots one. Raise it to eyeball segmentation on more fields. Default 3.
- **`src`** — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`target`** — (str) - Free-text label for the protein or marker imaged in channel_of_interest, e.g. 'GRA1'. The recruitment run prints it in its banner ('channel:3 = protein') to record what the recruitment ratio is measuring; it feeds no computation, so changing it alters nothing but that log line. Default 'protein'.
- **`target_intensity_min`** — (float) - Recruitment-analysis cutoff on the 95th-percentile intensity of channel_of_interest inside each cell: cells at or below it are discarded before recruitment ratios are computed. Raise it to keep only strongly expressing cells; set 0 or None to disable the filter entirely. Raw intensity units, default 1.
- **`treatment_plate_metadata`** — (list of lists) - Wells that received each entry of treatments, one inner list per treatment in the same order, e.g. [['r1','r2','r3'],['r4','r5','r6']]. Entries must start with 'r' (row) or 'c' (column); anything else is IGNORED and those wells get no treatment label rather than an error. Wells you do not list are still kept -- 'condition' joins whatever cell/pathogen/treatment labels exist, so an unlisted well simply carries fewer. Default None.
- **`treatments`** — (list) - Names of the drug or treatment conditions in the experiment, e.g. ['dmso','lovastatin']. Each name is written into the treatment column and folded into the combined condition label used for grouping and plotting; positionally paired with treatment_plate_metadata (or treatment_loc), which lists the wells for each. Default ['cm','lovastatin'].

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    'cell_chann_dim': 3,
    'cell_intensity_range': [0, 100000],
    'cell_mask_dim': 4,
    'cell_plate_metadata': None,
    'cell_size_range': [0, 100000],
    'cell_types': ['HeLa'],
    'cells_per_well': 0,
    'channel_dims': [0, 1, 2, 3],
    'channel_of_interest': 2,
    'figuresize': 10,
    'nuclei_limit': 1,
    'nucleus_chann_dim': 0,
    'nucleus_intensity_range': [0, 100000],
    'nucleus_mask_dim': 5,
    'nucleus_size_range': [0, 100000],
    'pathogen_chann_dim': 2,
    'pathogen_intensity_range': [0, 100000],
    'pathogen_limit': 10,
    'pathogen_mask_dim': 6,
    'pathogen_plate_metadata': [['c1', 'c2', 'c3'], ['c4', 'c5', 'c6']],
    'pathogen_size_range': [0, 100000],
    'pathogen_types': ['pathogen_1', 'pathogen_2'],
    'plot': True,
    'plot_control': True,
    'plot_nr': 3,
    'src': 'path',
    'target': 'protein',
    'target_intensity_min': 1,
    'treatment_plate_metadata': [['r1', 'r2', 'r3'], ['r4', 'r5', 'r6']],
    'treatments': ['cm', 'lovastatin'],
}

In [ ]:
analyze_recruitment(settings)

## Where the output went

Per-object recruitment ratios and per-condition summary plots.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.